In [1]:
import zipfile
import os

# Path to your uploaded zip file
zip_path = "/content/dentix_split.zip"
extract_path = "/content/dentix_split"

# Unzip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Check folder structure
os.listdir(extract_path)


['dentix_split']

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cpu')

In [4]:
# Paths
data_dir = extract_path

# Training transforms (with augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Validation transforms
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# Load datasets
# Adjust the path depending on what os.listdir showed
data_dir = "/content/dentix_split/dentix_split"  # <-- update if needed

train_dataset = datasets.ImageFolder(root=f"{data_dir}/train", transform=train_transforms)
val_dataset = datasets.ImageFolder(root=f"{data_dir}/val", transform=val_transforms)


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)


Classes: ['calculus', 'caries', 'gingivitis', 'hypodontia', 'mouth_ulcer', 'tooth_discoloration']


In [5]:
print(os.listdir(f"{data_dir}/train"))
print(os.listdir(f"{data_dir}/val"))


['hypodontia', 'mouth_ulcer', 'calculus', 'caries', 'tooth_discoloration', 'gingivitis']
['hypodontia', 'mouth_ulcer', 'calculus', 'caries', 'tooth_discoloration', 'gingivitis']


In [6]:
model = models.mobilenet_v2(pretrained=True)

# Freeze backbone
for param in model.features.parameters():
    param.requires_grad = False

# Replace classifier for our number of classes
model.classifier[1] = nn.Linear(model.last_channel, num_classes)
model = model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:01<00:00, 10.6MB/s]


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)


In [8]:
def train_model(model, epochs=10):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_acc = correct / total
        epoch_loss = running_loss / total

        # Validation
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_acc = val_correct / val_total

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {epoch_loss:.4f} "
              f"Train Acc: {train_acc:.2f} Val Acc: {val_acc:.2f}")


In [10]:
train_model(model, epochs=10)


Epoch [1/10] Loss: 0.7349 Train Acc: 0.73 Val Acc: 0.79
Epoch [2/10] Loss: 0.7061 Train Acc: 0.73 Val Acc: 0.79
Epoch [3/10] Loss: 0.7071 Train Acc: 0.73 Val Acc: 0.81
Epoch [4/10] Loss: 0.6985 Train Acc: 0.73 Val Acc: 0.82
Epoch [5/10] Loss: 0.6672 Train Acc: 0.75 Val Acc: 0.82
Epoch [6/10] Loss: 0.6682 Train Acc: 0.74 Val Acc: 0.82
Epoch [7/10] Loss: 0.6822 Train Acc: 0.74 Val Acc: 0.81
Epoch [8/10] Loss: 0.6934 Train Acc: 0.73 Val Acc: 0.81
Epoch [9/10] Loss: 0.6895 Train Acc: 0.74 Val Acc: 0.82
Epoch [10/10] Loss: 0.6665 Train Acc: 0.74 Val Acc: 0.82


In [12]:
torch.save({
    "model_state": model.state_dict(),
    "classes": class_names
}, "dentix_mobilenetv2_demo.pkl")


In [13]:
checkpoint = torch.load("dentix_mobilenetv2_demo.pth", map_location=device)
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, len(checkpoint["classes"]))
model.load_state_dict(checkpoint["model_state"])
model = model.to(device)
model.eval()

class_names = checkpoint["classes"]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [14]:
def predict_mobilenet(image_path):
    image = Image.open(image_path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        probs = torch.softmax(outputs, dim=1)
        conf, pred = torch.max(probs, 1)

    return class_names[pred.item()], conf.item()


In [23]:
image_path = "/content/dentix_split/dentix_split/val/tooth_discoloration/tooth_discoloration_124.jpg"  # upload a test image to Colab
pred, confidence = predict_mobilenet(image_path)
print(f"Disease: {pred}, Confidence: {confidence:.2f}")


Disease: tooth_discoloration, Confidence: 0.93
